<a href="https://colab.research.google.com/github/Bharathyramakrishnan/FullStackGenAI/blob/FullStack_GENAI/Non_instruction_Causal_LLM_Fine_tuning_or_Domain_Adaptive_Continued_Pretraining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Notebook 1: Pharma PDF → Raw Text → Non-Instruction Causal LLM Fine-Tuning
Non-instruction Causal LM Fine-tuning or Domain-Adaptive Continued Pretraining
Causal meaning: Past causes future prediction.
Goal: Use a pharma-domain PDF as raw text data and perform non-instruction causal language model fine-tuning using LoRA/QLoRA.

#### Pipeline
Pharma PDF
   ↓
PDF text extraction
   ↓
Text cleaning and normalization
   ↓
Data creation
   ↓
Hugging Face Dataset Conversion
   ↓
Tokenization
   ↓
LoRA/QLoRA fine-tuning
   ↓
Validation loss
   ↓
Adapter saving and reloading
   ↓
Text continuation inference

Continued Pretraining vs Instruction Fine-Tuning
In this notebook, we are performing continued pretraining / non-instruction fine-tuning on raw pharma PDF text.

The model is given raw domain text such as:

Metformin is one of the most widely prescribed oral antihyperglycemic agents...

The model then learns to predict the next token from this raw text.

This means the model learns:

Pharma language
Drug names
Medical terminology
Scientific writing style
Domain-specific sentence patterns
However, the model is not explicitly taught:

How to answer a user's question
How to follow instructions
How to respond in Q&A format
How to behave like a domain-specific chatbot

What Instruction Fine-Tuning Looks Like
In instruction fine-tuning, the training data is prepared in an instruction-response format.

Example:

{
  "instruction": "Explain the mechanism of action of Metformin.",
  "input": "",
  "output": "Metformin primarily activates AMPK, which improves glucose uptake and reduces hepatic gluconeogenesis."
}

{
  "messages": [
    {
      "role": "user",
      "content": "What is the primary mechanism of action of Metformin?"
    },
    {
      "role": "assistant",
      "content": "Metformin primarily works by activating AMPK..."
    }
  ]
}

Pipeline
Non-instrcution FT
      ↓
will save the model
      ↓
load the model
      ↓
I will perform instruction FT on same model
      ↓
will save our model
      ↓
again will load the same model
      ↓
will perform the preference tuning on top of it
   

Pipeline
Non-instrcution FT(RAW)
      ↓
will save the model
      ↓
load the model
      ↓
I will perform instruction FT on same model(question/answer data)
      ↓
will save our model
      ↓
again will load the same model
      ↓
will perform the preference tuning on top of it(choosed/ reject data)
   
we are going to train the LORA adapter

In [ ]:
# ============================================================
# 1. Install required libraries
# ============================================================
# PyMuPDF: PDF text extraction
# datasets: Hugging Face dataset creation
# transformers/accelerate: model, tokenizer, Trainer
# peft: LoRA/QLoRA adapters
# bitsandbytes: 4-bit/8-bit quantized loading

!pip install -q -U pymupdf datasets transformers accelerate peft bitsandbytes sentencepiece

In [ ]:
!pip uninstall -y pyarrow datasets
!pip install pyarrow==15.0.2
!pip install datasets==2.19.2

In [ ]:
# ============================================================
# 2. Imports
# ============================================================

import os
import re
import gc
import math
import json
import random
import unicodedata
from dataclasses import dataclass, asdict
from typing import List, Dict, Any

import fitz  # PyMuPDF
import torch
from datasets import Dataset, DatasetDict

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
    set_seed,
)

from peft import (
    LoraConfig,
    TaskType,
    get_peft_model,
    prepare_model_for_kbit_training,
    PeftModel,
)

1. dataclass
2. enum class
3. pydantic class
4. typedict class
5. abstract class

In [ ]:
# ============================================================
# 3. Global configuration
# ============================================================
# Keep all important parameters in one place.
# This makes the notebook easier to debug, reproduce, and productionize.

@dataclass
class Config:
    # Upload your pharma PDF to this path in Colab or update this value.
    pdf_path: str = "/content/Metformin-Lipid-Therapy-Knowledge.pdf"

    # TinyLlama is selected because it is lightweight for classroom use.
    model_name: str = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

    # Output directories.
    output_dir: str = "/content/pharma_tinyllama_lora_output"
    adapter_dir: str = "/content/pharma_tinyllama_lora_adapter"
    processed_data_dir: str = "/content/pharma_processed_data"

    # Text preprocessing.
    min_chars_per_paragraph: int = 80
    block_size: int = 512

    # Train/eval split.
    test_size: float = 0.15
    seed: int = 42

    # LoRA parameters.
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.05

    # Training parameters.
    num_train_epochs: float = 3.0
    per_device_train_batch_size: int = 1
    per_device_eval_batch_size: int = 1
    gradient_accumulation_steps: int = 8
    learning_rate: float = 2e-4
    warmup_ratio: float = 0.03
    weight_decay: float = 0.01
    logging_steps: int = 5
    eval_steps: int = 10
    save_steps: int = 25
    save_total_limit: int = 2
    # For a quick demo, set max_steps to 20 or 30.
    # For a complete run, keep it as -1.
    max_steps: int = -1

In [ ]:
config = Config()

In [ ]:
config

In [ ]:
print(json.dumps(asdict(config), indent=2))

In [ ]:
config.output_dir

In [ ]:
config.processed_data_dir

In [ ]:
os.makedirs(config.output_dir, exist_ok=True)
os.makedirs(config.adapter_dir, exist_ok=True)
os.makedirs(config.processed_data_dir, exist_ok=True)

In [ ]:
# ============================================================
# 4. Optional Colab upload helper
# ============================================================
# Run this cell only if your PDF is not already present at config.pdf_path.

if not os.path.exists(config.pdf_path):
    print(f"PDF not found at: {config.pdf_path}")
else:
    print(f"PDF found: {config.pdf_path}")

In [ ]:
# # ============================================================
# # 5. Extract text from PDF
# # ============================================================
def extract_pdf_pages(pdf_path: str) -> List[Dict[str, Any]]:
    # Extract page-level text from a PDF.
    pages = []
    with fitz.open(pdf_path) as doc:
        for page_index, page in enumerate(doc, start=1):
            text = page.get_text("text")
            text = text.strip() if text else ""
            if text:
                pages.append({
                    "page": page_index,
                    "text": text,
                    "char_count": len(text),
                })
    return pages


In [ ]:
config.pdf_path

In [ ]:
pdf_pages = extract_pdf_pages(config.pdf_path)

In [ ]:
print(f"Total pages with extracted text: {len(pdf_pages)}")
print("Page-level character counts:")
for item in pdf_pages:
    print(f"Page {item['page']}: {item['char_count']} characters")

In [ ]:
print(pdf_pages[0]["text"])

# ============================================================
# 6. Text cleaning utilities
# ============================================================

| Cleaning Step                          | Code / Logic                             | What It Does                                                                  | Example Before                                                  | Example After                                                  | Why It Matters for Fine-Tuning                                            |
| -------------------------------------- | ---------------------------------------- | ----------------------------------------------------------------------------- | --------------------------------------------------------------- | -------------------------------------------------------------- | ------------------------------------------------------------------------- |
| Unicode normalization                  | `unicodedata.normalize("NFKC", text)`    | Converts unusual Unicode characters into standard readable characters.        | `ＡＭＰＫ`, `ﬁ`                                                     | `AMPK`, `fi`                                                   | Prevents tokenizer confusion caused by hidden or non-standard characters. |
| Remove zero-width characters           | `text.replace("\u200b", "")`             | Removes invisible zero-width spaces from PDF text.                            | `Metformin​ activates AMPK`                                     | `Metformin activates AMPK`                                     | Invisible characters can create bad tokens and noisy training data.       |
| Remove BOM / hidden marker             | `text.replace("\ufeff", "")`             | Removes hidden Byte Order Mark characters sometimes found in extracted text.  | `﻿Metformin is used...`                                         | `Metformin is used...`                                         | Keeps the training text clean and consistent.                             |
| Fix hyphenated line breaks             | `re.sub(r"(\w)-\n(\w)", r"\1\2", text)`  | Joins words that were broken across PDF lines.                                | `gluconeogene-\nsis`                                            | `gluconeogenesis`                                              | Prevents the model from learning broken medical terms.                    |
| Normalize spaces and tabs              | `re.sub(r"[ \t]+", " ", text)`           | Converts multiple spaces or tabs into one space.                              | `Metformin     activates    AMPK`                               | `Metformin activates AMPK`                                     | Makes text consistent and easier for tokenizer/model to learn.            |
| Normalize blank lines                  | `re.sub(r"\n{3,}", "\n\n", text)`        | Converts too many blank lines into a proper paragraph gap.                    | `Para 1\n\n\n\nPara 2`                                          | `Para 1\n\nPara 2`                                             | Preserves paragraph structure without unnecessary whitespace noise.       |
| Remove standalone page numbers         | `re.sub(r"(?m)^\s*\d+\s*$", "", text)`   | Removes lines that contain only page numbers.                                 | `1` or `23`                                                     | Removed                                                        | Prevents the model from learning irrelevant PDF page numbers.             |
| Split into paragraphs                  | `re.split(r"\n\s*\n", text)`             | Splits text wherever there is a blank line.                                   | `Para 1\n\nPara 2`                                              | `["Para 1", "Para 2"]`                                         | Helps preserve meaningful document structure.                             |
| Remove line wrapping inside paragraphs | `re.sub(r"\n+", " ", paragraph)`         | Converts broken lines inside the same paragraph into a single paragraph line. | `Metformin is widely prescribed\noral antihyperglycemic agent.` | `Metformin is widely prescribed oral antihyperglycemic agent.` | Prevents the model from learning artificial PDF line breaks.              |
| Normalize paragraph spacing            | `re.sub(r"\s+", " ", paragraph).strip()` | Removes extra spaces inside each paragraph and trims start/end spaces.        | `  Metformin   activates   AMPK.  `                             | `Metformin activates AMPK.`                                    | Produces clean, readable training examples.                               |
| Remove empty paragraphs                | `if paragraph:`                          | Keeps only non-empty cleaned paragraphs.                                      | `""`                                                            | Removed                                                        | Avoids useless blank samples in the dataset.                              |
| Rebuild cleaned text                   | `"\n\n".join(cleaned_paragraphs)`        | Joins cleaned paragraphs with two newlines.                                   | List of cleaned paragraphs                                      | Clean paragraph-level text                                     | Creates a clean corpus suitable for causal LM training.                   |
| Track cleaned page length              | `char_count: len(cleaned_text)`          | Stores number of characters after cleaning.                                   | Raw page length unknown                                         | `char_count = 1450`                                            | Helps debug whether a page has too little or too much extracted content.  |
| Preview cleaned output                 | `cleaned_pages[0]["text"][:1500]`        | Prints first 1500 characters of cleaned page 1.                               | Full cleaned page                                               | Preview text                                                   | Helps manually verify that cleaning worked correctly.                     |


In [ ]:
# ============================================================
# 6. Text cleaning utilities
# ============================================================

In [ ]:
# Ye hidden/weird Unicode characters ko normal form me convert karta hai.

# Example:

# ＡＭＰＫ → AMPK
# ﬁ → fi
def normalize_unicode(text: str) -> str:
    # Normalize Unicode characters to reduce hidden character issues.
    return unicodedata.normalize("NFKC", text)

In [ ]:
def clean_pdf_text(text: str) -> str:
    # Clean raw PDF text while preserving meaningful scientific content.
    text = normalize_unicode(text)

    # Remove zero-width and non-printing characters often found in PDFs.
    # Ye special hidden characters remove karta hai.
    # \u200b
    # Zero-width space hota hai. Ye dikhta nahi hai but text ke andar hota hai.
    # \ufeff
    # Byte Order Mark / hidden marker hota hai. PDF/text extraction me kabhi aa jata hai.
    # Inhe remove karna important hai, warna tokenizer confuse ho sakta hai.
    text = text.replace("\u200b", "")
    text = text.replace("\ufeff", "")

    # Fix hyphenated line breaks: for example, "gluconeogene-\nsis" -> "gluconeogenesis".
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)

    # Convert multiple spaces/tabs into a single space.
    text = re.sub(r"[ \t]+", " ", text)

    # Normalize repeated blank lines.
    text = re.sub(r"\n{3,}", "\n\n", text)

    # Remove standalone page numbers.
    text = re.sub(r"(?m)^\s*\d+\s*$", "", text)

    # Preserve paragraph boundaries, but remove line wrapping inside paragraphs.
    paragraphs = re.split(r"\n\s*\n", text)

    cleaned_paragraphs = []

    for paragraph in paragraphs:
        paragraph = re.sub(r"\n+", " ", paragraph)
        paragraph = re.sub(r"\s+", " ", paragraph).strip()
        if paragraph:
            cleaned_paragraphs.append(paragraph)

    return "\n\n".join(cleaned_paragraphs)

In [ ]:
pdf_pages

In [ ]:
cleaned_pages = []

In [ ]:
for page in pdf_pages:
    cleaned_text = clean_pdf_text(page["text"])
    cleaned_pages.append({
        "page": page["page"],
        "text": cleaned_text,
        "char_count": len(cleaned_text),
    })

In [ ]:
cleaned_pages

In [ ]:
print("Cleaned page preview:\n")
print(cleaned_pages[0]["text"])

In [ ]:
cleaned_pages

In [ ]:
len("Split cleaned pages into paragraphs")

In [ ]:
# ============================================================
# 7. Split cleaned pages into paragraphs
# ============================================================
# Paragraph records are clean units for audit and inspection.
# Later, tokenized text will be packed into fixed-size training blocks.

def split_into_paragraph_records(cleaned_pages: List[Dict[str, Any]], min_chars: int = 80) -> List[Dict[str, Any]]:
    records = []
    seen = set()

    for page in cleaned_pages:
        paragraphs = re.split(r"\n\s*\n", page["text"])
        for paragraph_index, paragraph in enumerate(paragraphs, start=1):
            paragraph = paragraph.strip()
            if len(paragraph) < min_chars:
                continue

            # Deduplicate exact repeated paragraphs.
            normalized_key = re.sub(r"\s+", " ", paragraph.lower()).strip()
            if normalized_key in seen:
                continue
            seen.add(normalized_key)

            records.append({
                "text": paragraph,
                "source_page": page["page"],
                "paragraph_id": paragraph_index,
                "char_count": len(paragraph),
            })

    return records

In [ ]:
config.min_chars_per_paragraph

In [ ]:
paragraph_records = split_into_paragraph_records(cleaned_pages, min_chars=config.min_chars_per_paragraph)

In [ ]:
print(f"Total paragraph records: {len(paragraph_records)}")
for i, record in enumerate(paragraph_records[:5]):
    print("=" * 100)
    print(f"Record {i} | Page {record['source_page']} | Characters: {record['char_count']}")
    print(record["text"])

In [ ]:
paragraph_records

In [ ]:
config.processed_data_dir

In [ ]:
config.processed_data_dir

In [ ]:
# ============================================================
# 8. Save extracted and cleaned corpus for auditability
# ============================================================
# In real projects, always save intermediate datasets.
# This helps with reproducibility, debugging, and compliance review.

raw_pages_path = os.path.join(config.processed_data_dir, "pdf_pages_raw.jsonl")
paragraphs_path = os.path.join(config.processed_data_dir, "pharma_paragraph_process.jsonl")

with open(raw_pages_path, "w", encoding="utf-8") as f:
    for item in pdf_pages:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

with open(paragraphs_path, "w", encoding="utf-8") as f:
    for item in paragraph_records:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"Saved raw pages to: {raw_pages_path}")
print(f"Saved cleaned paragraph corpus to: {paragraphs_path}")

In [ ]:
# ============================================================
# 9. Create Hugging Face Dataset
# ============================================================

if len(paragraph_records) < 2:
    raise ValueError(
        "The extracted corpus is too small. Please provide a larger pharma PDF or lower min_chars_per_paragraph."
    )

text_dataset = Dataset.from_list(paragraph_records)


In [ ]:
print(text_dataset)

In [ ]:
print(text_dataset[0])

In [ ]:
# ============================================================
# 10. Train/eval split
# ============================================================
# Even for small demos, keep an evaluation set.
# This gives us validation loss and perplexity.

split_dataset = text_dataset.train_test_split(test_size=config.test_size, seed=config.seed)

from datasets import DatasetDict
dataset = DatasetDict({
    "train": split_dataset["train"],
    "validation": split_dataset["test"],
})

print(dataset)

In [ ]:
## 11. Load tokenizer

The tokenizer converts text into token IDs.

For causal language modeling, the model learns:

In [ ]:
# ============================================================
# 11. Load tokenizer
# ============================================================

from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(config.model_name, use_fast=True)

# Some Llama-style models do not define a pad token.
# For causal LM fine-tuning, using EOS as PAD is a common practical choice.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

In [ ]:
print(f"Tokenizer loaded: {config.model_name}")
print(f"Vocab size: {len(tokenizer)}")
print(f"Pad token: {tokenizer.pad_token} | Pad token id: {tokenizer.pad_token_id}")
print(f"EOS token: {tokenizer.eos_token} | EOS token id: {tokenizer.eos_token_id}")

In [ ]:
# ============================================================
# 12. Tokenization and text packing
# ============================================================
def tokenize_function(examples):
    # Tokenize text without padding. Padding is handled dynamically by the collator.
    return tokenizer(examples["text"])

In [ ]:
tokenized_datasets = dataset.map(
    tokenize_function,
    remove_columns=dataset["train"].column_names,
    desc="Tokenizing text corpus",
)

| Parameter                                       | Meaning                                                                                                                               |
| ----------------------------------------------- | ------------------------------------------------------------------------------------------------------------------------------------- |
| `tokenize_function`                             | This function converts each text example into token IDs.                                                                              |
| `batched=True`                                  | The function processes multiple rows at once instead of one row at a time. This makes tokenization faster.                            |
| `remove_columns=datasets["train"].column_names` | After tokenization, the original dataset columns are removed. Only tokenized columns such as `input_ids` and `attention_mask` remain. |
| `desc="Tokenizing text corpus"`                 | This message is shown in the progress bar so we can understand that tokenization is currently running.                                |


In [ ]:
tokenized_datasets

In [ ]:
tokenized_datasets['train']['input_ids'][0]

def tokenize_fn(examples): tokens = tokenizer( examples["text"], truncation=True, padding="max_length", max_length=512 ) tokens["labels"] = tokens["input_ids"].copy() return tokens

tokenized = dataset.map(tokenize_fn, batched=True, remove_columns=["text"])

trainer = Trainer( model=model, args=training_args, train_dataset=tokenized )

Approach 1: Without text packing

Each paragraph/example ko directly 512 token length me convert kar diya:

Paragraph 1 → tokenize → pad/truncate to 512 tokens

Paragraph 2 → tokenize → pad/truncate to 512 tokens

Paragraph 3 → tokenize → pad/truncate to 512 tokens

Example:

Paragraph has 100 real tokens

Padding added = 412 tokens

Final length = 512 tokens

Good for:

Beginner teaching

Small demo

Simple notebook

Less complex explanation

Problem:

Lots of padding

GPU wastage

Less efficient training

All tokenized text ko join karke fixed blocks banata hai:

Paragraph 1 tokens + Paragraph 2 tokens + Paragraph 3 tokens ↓ One long token stream ↓ Split into 512-token blocks

Example:

Paragraph 1 = 100 tokens

Paragraph 2 = 150 tokens

Paragraph 3 = 262 tokens

Together = 512 real tokens

Yaha padding waste nahi hota.

Good for:

Better GPU utilization

More efficient continued pretraining

More real tokens per batch

Industry-style causal LM pretraining

In [ ]:
def create_training_blocks(tokenized_examples):
    # Join all token IDs from multiple examples into one long list.
    all_input_ids = []
    all_attention_masks = []

    for input_ids in tokenized_examples["input_ids"]:
        all_input_ids.extend(input_ids)

    for attention_mask in tokenized_examples["attention_mask"]:
        all_attention_masks.extend(attention_mask)

    # Calculate how many complete blocks we can create.
    total_tokens = len(all_input_ids)
    usable_tokens = (total_tokens // config.block_size) * config.block_size

    # If we do not have enough tokens to create even one block, return empty data.
    if usable_tokens == 0:
        return {
            "input_ids": [],
            "attention_mask": [],
            "labels": [],
        }

    # Keep only tokens that can fit into complete fixed-size blocks.
    all_input_ids = all_input_ids[:usable_tokens]
    all_attention_masks = all_attention_masks[:usable_tokens]

    # Split the long token list into fixed-size training blocks.
    input_id_blocks = []
    attention_mask_blocks = []

    for start_index in range(0, usable_tokens, config.block_size):
        end_index = start_index + config.block_size

        input_id_blocks.append(all_input_ids[start_index:end_index])
        attention_mask_blocks.append(all_attention_masks[start_index:end_index])

    # For causal language modeling, labels are the same as input IDs.
    # The model uses these labels to learn next-token prediction.
    labels = input_id_blocks.copy()

    return {
        "input_ids": input_id_blocks,
        "attention_mask": attention_mask_blocks,
        "labels": labels,
    }

In [ ]:
final_dataset = tokenized_datasets.map(
    create_training_blocks,
    batched=True,
    desc=f"Creating fixed-size training blocks of {config.block_size} tokens",
)

## What Does This Function Do?

This function converts tokenized text into **fixed-size training blocks**.

First, it joins all token IDs into one long sequence.  
Then, it cuts that long sequence into equal blocks of `config.block_size` tokens.

For causal language modeling, the labels are copied from `input_ids` because the model learns to predict the next token.

---

## Example

Suppose we have these tokenized inputs:

```text
Input token lists:

[10, 20, 30]
[40, 50]
[60, 70, 80, 90]

After joining all token lists together:

[10, 20, 30, 40, 50, 60, 70, 80, 90]

If:

block_size = 4

Then the final training blocks become:

Block 1 = [10, 20, 30, 40]
Block 2 = [50, 60, 70, 80]

The remaining token is:

[90]

This token is dropped because it cannot form a complete block of 4 tokens.

Simple Summary

This step prepares the final causal language modeling dataset by converting many small tokenized examples into equal-length token blocks.

In [ ]:
sample = final_dataset["train"][0]

In [ ]:
print("Keys:", sample.keys())
print("input_ids length:", len(sample["input_ids"]))
print("labels length:", len(sample["labels"]))
print("Decoded sample preview:\n")
print(tokenizer.decode(sample["input_ids"][:250]))

## 13. Load Model for QLoRA Training

In this step, we load the base model for fine-tuning.

If GPU is available, we load the model in **4-bit mode**.

This helps because:

- It uses less GPU memory
- It allows us to fine-tune larger models on limited hardware
- It is useful for Colab or small GPU environments
- It works well with LoRA/QLoRA fine-tuning

If GPU is not available, the model will load normally on CPU, but training will be much slower.

In [ ]:
# ============================================================
# 13. Load base model
# ============================================================
import torch
use_cuda = torch.cuda.is_available()
print("CUDA available:", use_cuda)
if use_cuda:
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# Clear memory before loading the model.
import gc
gc.collect()
if use_cuda:
    torch.cuda.empty_cache()

In [ ]:
from transformers import AutoModelForCausalLM

if use_cuda:
    from transformers import BitsAndBytesConfig
    from peft import prepare_model_for_kbit_training

    # Configure 4-bit quantization to reduce GPU memory usage.
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    # Load the base model in 4-bit mode on available GPU devices.
    base_model = AutoModelForCausalLM.from_pretrained(
        config.model_name,
        quantization_config=quantization_config,
        device_map="auto",
        trust_remote_code=True,
    )

    # Prepare the quantized model for stable LoRA/QLoRA training.
    base_model = prepare_model_for_kbit_training(base_model)

else:
    # Load the base model normally when GPU is not available.
    base_model = AutoModelForCausalLM.from_pretrained(
        config.model_name,
        torch_dtype=torch.float32,
        trust_remote_code=True,
    )

# Disable cache during training to reduce memory usage and avoid training warnings.
base_model.config.use_cache = False

print("Base model loaded successfully.")

In [ ]:
# ============================================================
# 14. Apply LoRA adapters
# ============================================================
# LoRA trains a small number of adapter parameters instead of updating all base model weights.
# This is cheaper than full fine-tuning and is widely used in real projects.
from peft import LoraConfig
from peft import TaskType

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=config.lora_r,
    lora_alpha=config.lora_alpha,
    lora_dropout=config.lora_dropout,
    bias="none",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)


In [ ]:
from peft import get_peft_model
model = get_peft_model(base_model, lora_config)

In [ ]:
model.print_trainable_parameters()

In [ ]:
# ============================================================
# 15. Data collator
# ============================================================
from transformers import DataCollatorForLanguageModeling
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

## Why Do We Need `DataCollatorForLanguageModeling`?

After tokenization and text packing, our dataset contains token IDs in a training-ready structure.

However, the `Trainer` still needs a component that can take multiple examples from the dataset and convert them into a proper batch during training.

That component is called a **data collator**.

```python
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)
What Does the Data Collator Do?

The data collator prepares mini-batches for the model.

It handles things like:

Collecting multiple training examples together
Padding sequences if required
Converting examples into tensors
Preparing labels for language modeling
Example

Suppose our packed dataset has training examples like this:

Example 1 = 512 tokens
Example 2 = 512 tokens
Example 3 = 512 tokens

During training, the Trainer may take two examples at a time:

Batch = Example 1 + Example 2

The data collator converts them into tensors like:

input_ids shape      = [2, 512]
attention_mask shape = [2, 512]
labels shape         = [2, 512]

This is the format the model expects during training.

Why mlm=False?

mlm means Masked Language Modeling.

Masked Language Modeling is used for BERT-style models.

Example:

Metformin is used for [MASK].

The model predicts the masked word:

diabetes

But we are using TinyLlama, which is a causal language model.

Causal language models learn by predicting the next token from left to right.

Example:

Metformin → is
Metformin is → used
Metformin is used → for
Metformin is used for → diabetes

So we set:

mlm=False

This tells Hugging Face:

Do not use BERT-style masked language modeling. Use causal language modeling instead.

Why Is This Needed Even After Tokenization and Packing?
Tokenization converts text into token IDs.

Text packing groups token IDs into fixed-size blocks.

But the data collator prepares those blocks into actual training batches.

So the flow is:

Raw pharma text
   ↓
Tokenization
   ↓
Token IDs
   ↓
Text packing
   ↓
Fixed-size training blocks
   ↓
Data collator
   ↓
Mini-batches for Trainer
   ↓
Model training

In [ ]:
# ============================================================
# 16. Training arguments
# ============================================================
# These settings are designed for a small classroom/demo run.
# For larger corpora, increase dataset size, epochs, and evaluation frequency carefully.

import inspect
from transformers import TrainingArguments

In [ ]:
training_kwargs = dict(
    output_dir=config.output_dir,
    num_train_epochs=config.num_train_epochs,
    max_steps=config.max_steps,
    per_device_train_batch_size=config.per_device_train_batch_size,
    per_device_eval_batch_size=config.per_device_eval_batch_size,
    gradient_accumulation_steps=config.gradient_accumulation_steps,
    learning_rate=config.learning_rate,
    warmup_steps=5,
    weight_decay=config.weight_decay,

    # Log training loss at every step for small demo datasets.
    logging_steps=1,
    logging_first_step=True,

    eval_steps=config.eval_steps,
    save_steps=config.save_steps,
    save_total_limit=config.save_total_limit,
    fp16=use_cuda,
    bf16=False,
    report_to="none",
    remove_unused_columns=False,
)

In [ ]:
from transformers import TrainingArguments
training_args = TrainingArguments(**training_kwargs)

In [ ]:
# ============================================================
# 17. Build Trainer
# ============================================================
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=final_dataset["train"],
    eval_dataset=final_dataset["validation"],
    data_collator=data_collator,
)
print("Trainer is ready.")

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# ============================================================
# 18. Start training
# ============================================================
train_result = trainer.train()
print("Training completed.")

In [ ]:
for log in trainer.state.log_history:
    print(log)

In [ ]:
# ============================================================
# 19. Save adapter and tokenizer
# ============================================================
trainer.model.save_pretrained(config.adapter_dir)
tokenizer.save_pretrained(config.adapter_dir)

In [ ]:
print(f"LoRA adapter saved to: {config.adapter_dir}")
print("Saved files:")
print(os.listdir(config.adapter_dir))

In [ ]:
# ============================================================
# 20. Push LoRA adapter to Hugging Face Hub
# ============================================================
repo_id = "BharathyRamakrishnan/pharma-tinyllama-domain-lora"

In [ ]:
from huggingface_hub import login
from google.colab import userdata

token = userdata.get("HF_TOKEN_WRITE")

login(token=token)

In [ ]:
from huggingface_hub import whoami

print(whoami())

In [ ]:
model.push_to_hub(
    repo_id,
    private=True
)

In [ ]:


from google.colab import userdata

token = userdata.get("HF_TOKEN_WRITE")

print(token is not None)
print(len(token) if token else "No token")

In [ ]:
tokenizer.push_to_hub(
    repo_id,
    private=True
)

In [ ]:
# ============================================================
# 21. Reload base model + LoRA adapter correctly
# ============================================================
# Clean old objects to free memory.

del trainer

try:
    del model
    del base_model
except NameError:
    pass

gc.collect()

if use_cuda:
    torch.cuda.empty_cache()

In [ ]:
from transformers import AutoTokenizer
inference_tokenizer = AutoTokenizer.from_pretrained(config.adapter_dir, use_fast=True)

if inference_tokenizer.pad_token is None:
    inference_tokenizer.pad_token = inference_tokenizer.eos_token

In [ ]:
if use_cuda:
    inference_base_model = AutoModelForCausalLM.from_pretrained(
        config.model_name,
        quantization_config=BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        ),
        device_map="auto",
        trust_remote_code=True,
    )
else:
    inference_base_model = AutoModelForCausalLM.from_pretrained(
        config.model_name,
        torch_dtype=torch.float32,
        trust_remote_code=True,
    )

In [ ]:
from peft import PeftModel
inference_model = PeftModel.from_pretrained(inference_base_model, config.adapter_dir)

inference_model.eval()

print("Base model + LoRA adapter loaded successfully for inference.")

In [ ]:
# ============================================================
# 22. Inference helper
# ============================================================
# Since this is non-instruction fine-tuning, prompts should look like text continuations,
# not chat-style questions.

def generate_completion(prompt: str, max_new_tokens: int = 120) -> str:
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # Convert prompt text into token IDs.
    inputs = inference_tokenizer(prompt, return_tensors="pt").to(device)

    # Generate text without calculating gradients because we are doing inference, not training.
    with torch.no_grad():
        outputs = inference_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=inference_tokenizer.eos_token_id,
        )

    # Convert generated token IDs back into readable text.
    return inference_tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
# ============================================================
# 23. Test text continuation
# ============================================================
# These prompts are continuation-style prompts.
# In Notebook 2, we will create instruction prompts for Q&A.

prompts = [
    "Metformin is one of the most widely prescribed oral antihyperglycemic agents",
    "Clinical trials have demonstrated that combining Atorvastatin with Ezetimibe",
    "Artificial intelligence is transforming pharmaceutical research by accelerating",
]


In [ ]:
# ============================================================
# 23. Test text continuation
# ============================================================

for prompt in prompts:
    print("=" * 100)
    print("PROMPT:")
    print(prompt)
    print("\nMODEL CONTINUATION:")
    print(generate_completion(prompt, max_new_tokens=120))
    print()

In [ ]:
# ============================================================
# 24. Optional merge step
# ============================================================
# This step merges the LoRA adapter into the base model.
# Use this only when you want a standalone model for deployment.

import os
import torch
from transformers import AutoModelForCausalLM
from peft import PeftModel

merged_model_dir = "/content/pharma_tinyllama_merged_model"
os.makedirs(merged_model_dir, exist_ok=True)

In [ ]:
# Reload the base model in float16 for safe merging.
base_model = AutoModelForCausalLM.from_pretrained(
    config.model_name,
    dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
    trust_remote_code=True,
)

In [ ]:
import torchao
print(torchao.__version__)

In [ ]:
# Load the trained LoRA adapter on top of the base model.
model_with_adapter = PeftModel.from_pretrained(
    base_model,
    config.adapter_dir
)

In [ ]:
!pip install torchao>=0.16.0

In [ ]:
import torch
import torchao

print(torch.__version__)
print(torchao.__version__)

In [ ]:
# Merge LoRA adapter weights into the base model weights.
merged_model = model_with_adapter.merge_and_unload()

In [ ]:
# Save the merged standalone model and tokenizer.

merged_model.save_pretrained(merged_model_dir)

inference_tokenizer.save_pretrained(merged_model_dir)

print(f"Merged model saved to: {merged_model_dir}")

Stage 2: Continue with Instruction Fine-Tuning on the Same Domain-Adapted Finetuned Model

In Stage 1, we performed non-instruction fine-tuning / domain-adaptive continued pretraining on raw pharma PDF text.

Now we continue from the same Stage 1 LoRA adapter and perform instruction fine-tuning using structured pharma instruction-response examples.
Base TinyLlama
   ↓
Stage 1: Raw pharma text continued pretraining using LoRA
   ↓
Stage 1 domain-adapted LoRA adapter
   ↓
Stage 2: Instruction fine-tuning on pharma Q&A data
   ↓
Final instruction-tuned pharma LoRA adapter
This means we are not starting from scratch. We are continuing from the model adapter trained in the previous stage.

What changes in instruction fine-tuning?

For non-instruction fine-tuning, the data looked like raw text:

Metformin is one of the most widely prescribed oral antihyperglycemic agents...
For instruction fine-tuning, the data looks like:

{
  "instruction": "Explain the mechanism of action of Metformin.",
  "input": "",
  "output": "Metformin primarily activates AMPK..."
}
This teaches the model not only pharma language, but also how to answer user instructions.

In [ ]:
instruction_data_path = "/content/pharma_instruction_dataset.jsonl"

In [ ]:
from datasets import load_dataset

In [ ]:
instruction_dataset = load_dataset(
    "json",
    data_files=instruction_data_path,
    split="train"
)

In [ ]:
print(instruction_dataset)

In [ ]:
print(instruction_dataset[0])

In [ ]:
# ============================================================
# Format instruction records
# ============================================================
# We convert every record into Alpaca-style training text.

def format_instruction_record(record):
    instruction = str(record.get("instruction", "")).strip()
    input_text = str(record.get("input", "")).strip()
    output_text = str(record.get("output", "")).strip()

    if input_text:
        text = (
            f"### Instruction:\n{instruction}\n\n"
            f"### Input:\n{input_text}\n\n"
            f"### Response:\n{output_text}"
        )
    else:
        text = (
            f"### Instruction:\n{instruction}\n\n"
            f"### Response:\n{output_text}"
        )

    return {"text": text}


instruction_dataset = instruction_dataset.map(format_instruction_record)

print(instruction_dataset[0]["text"])

In [ ]:
# ============================================================
# Create train-validation split
# ============================================================

instruction_datasets = instruction_dataset.train_test_split(
    test_size=0.15,
    seed=42
)

instruction_datasets["validation"] = instruction_datasets.pop("test")

print(instruction_datasets)
print("Train examples:", len(instruction_datasets["train"]))
print("Validation examples:", len(instruction_datasets["validation"]))

In [ ]:
# ============================================================
# Tokenize instruction dataset
# ============================================================
# The tokenizer converts text into token IDs for model training.

from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(config.model_name, use_fast=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(tokenizer.pad_token)
instruction_max_length = 512

In [ ]:
def tokenize_instruction_function(examples):
    tokens = tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=instruction_max_length,
    )

    # For causal LM, labels are copied from input_ids.
    tokens["labels"] = tokens["input_ids"].copy()

    # Ignore padding tokens in the loss calculation.
    tokens["labels"] = [
        [
            token if mask == 1 else -100
            for token, mask in zip(input_ids, attention_mask)
        ]
        for input_ids, attention_mask in zip(tokens["input_ids"], tokens["attention_mask"])
    ]

    return tokens

In [ ]:
When we tokenize instruction data, all examples are not the same length.

Example:

Example 1 = 20 tokens Example 2 = 80 tokens Example 3 = 150 tokens

But for training, we often make every example the same length, like:

max_length = 512

So shorter examples get extra padding tokens.

Example:

Real text tokens + padding tokens = 512 tokens

Now the problem is:

We want the model to learn from real text, not from padding.

So we use -100 in labels.

-100 tells PyTorch:

Ignore this position while calculating loss.

In [ ]:
instruction_tokenized_datasets = instruction_datasets.map(
    tokenize_instruction_function,
    batched=True,
    remove_columns=instruction_datasets["train"].column_names,
    desc="Tokenizing instruction dataset",
)

print(instruction_tokenized_datasets)

In [ ]:
# # ============================================================
# # Reload Stage 1 LoRA adapter as trainable
# # ============================================================
# # We continue instruction fine-tuning from the non-instruction LoRA adapter.

# gc.collect()

# if torch.cuda.is_available():
#     torch.cuda.empty_cache()

# use_cuda = torch.cuda.is_available()

# if use_cuda:
#     instruction_base_model = AutoModelForCausalLM.from_pretrained(
#         config.model_name,
#         quantization_config=BitsAndBytesConfig(
#             load_in_4bit=True,
#             bnb_4bit_quant_type="nf4",
#             bnb_4bit_compute_dtype=torch.float16,
#             bnb_4bit_use_double_quant=True,
#         ),
#         device_map="auto",
#         trust_remote_code=True,
#     )

#     instruction_base_model = prepare_model_for_kbit_training(instruction_base_model)

# else:
#     instruction_base_model = AutoModelForCausalLM.from_pretrained(
#         config.model_name,
#         torch_dtype=torch.float32,
#         trust_remote_code=True,
#     )

# instruction_base_model.config.use_cache = False

# # Load the Stage 1 adapter and keep it trainable for Stage 2.
# instruction_model = PeftModel.from_pretrained(
#     instruction_base_model,
#     config.adapter_dir,
#     is_trainable=True,
# )

# instruction_model.print_trainable_parameters()

# Base model
#    +
# Stage 1 domain LoRA adapter
#    ↓ continue training
# Stage 1 + Stage 2 final LoRA adapter

| Point                          | Approach 1: Continue Same Stage 1 LoRA Adapter                                         | Approach 2: Merge Stage 1, Then Add New LoRA Adapter                                                      |
| ------------------------------ | -------------------------------------------------------------------------------------- | --------------------------------------------------------------------------------------------------------- |
| Flow                           | Base model + Stage 1 LoRA adapter → continue training same adapter on instruction data | Base model + Stage 1 LoRA adapter → merge → load merged model → add new LoRA adapter for instruction data |
| Main idea                      | The same adapter learns both domain language and instruction-following behavior        | Stage 1 knowledge becomes part of the merged model, then a new adapter learns instruction behavior        |
| Simplicity                     | Easier for students to understand                                                      | More complex because merge step is involved                                                               |
| Merge required before Stage 2? | No                                                                                     | Yes                                                                                                       |
| Risk/complexity                | Lower complexity                                                                       | Higher complexity, especially if Stage 1 model was loaded in 4-bit/QLoRA mode                             |
| Output size                    | Small final LoRA adapter                                                               | Large merged Stage 1 model + small Stage 2 adapter                                                        |
| Hugging Face upload            | Easy, only adapter can be pushed                                                       | Heavier because merged model is large                                                                     |
| Best for course/demo           | Best choice                                                                            | Use only if you already merged Stage 1                                                                    |
| Best for production            | Good for experimentation and adapter-based deployment                                  | Useful when you want Stage 1 knowledge permanently inside the base model                                  |
| Recommended for your notebook? | **Yes, recommended**                                                                   | Only if Stage 1 adapter is already merged                                                                 |


In [ ]:
# ============================================================
# Load merged Stage 1 model and add new LoRA adapter for instruction tuning
# ============================================================

# Merged Stage 1 Model
#    +
# New LoRA adapter for instruction tuning

import gc
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

use_cuda = torch.cuda.is_available()

merged_model_dir = "/content/pharma_tinyllama_merged_model"

if use_cuda:
    # Load merged Stage 1 model in 4-bit mode for QLoRA instruction tuning.
    instruction_base_model = AutoModelForCausalLM.from_pretrained(
        merged_model_dir,
        quantization_config=BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        ),
        device_map="auto",
        trust_remote_code=True,
    )

    instruction_base_model = prepare_model_for_kbit_training(instruction_base_model)

else:
    # CPU fallback. Training on CPU will be slow.
    instruction_base_model = AutoModelForCausalLM.from_pretrained(
        merged_model_dir,
        torch_dtype=torch.float32,
        trust_remote_code=True,
    )

instruction_base_model.config.use_cache = False

# Create a new LoRA adapter for instruction fine-tuning.
instruction_lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

instruction_model = get_peft_model(
    instruction_base_model,
    instruction_lora_config
)

instruction_model.print_trainable_parameters()

In [ ]:
# ============================================================
# Instruction fine-tuning data collator
# ============================================================
# This prepares mini-batches for causal language model training.

from transformers import DataCollatorForLanguageModeling
instruction_data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

In [ ]:
# ============================================================
# Instruction fine-tuning arguments
# ============================================================

instruction_output_dir = "/content/pharma_tinyllama_instruction_lora_output"
instruction_adapter_dir = "/content/pharma_tinyllama_instruction_lora_adapter"

os.makedirs(instruction_output_dir, exist_ok=True)
os.makedirs(instruction_adapter_dir, exist_ok=True)

In [ ]:
from transformers import TrainingArguments

instruction_training_args = TrainingArguments(
    output_dir=instruction_output_dir,

    # Train for 5 full epochs.
    num_train_epochs=5,
    max_steps=-1,

    # Batch settings.
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,

    # Optimizer settings.
    learning_rate=1e-4,
    warmup_steps=5,
    weight_decay=0.01,

    # Show training loss at every step.
    logging_steps=1,
    logging_first_step=True,

    # Run validation at every step.
    eval_strategy="steps",
    eval_steps=1,

    # Save checkpoints.
    save_steps=25,
    save_total_limit=2,

    # Precision settings.
    fp16=use_cuda,
    bf16=False,

    # Disable external logging tools.
    report_to="none",

    # Keep required columns.
    remove_unused_columns=False,
)

print(instruction_training_args)

In [ ]:
# ============================================================
# Build instruction Trainer
# ============================================================

from transformers import Trainer
instruction_trainer = Trainer(
    model=instruction_model,
    args=instruction_training_args,
    train_dataset=instruction_tokenized_datasets["train"],
    eval_dataset=instruction_tokenized_datasets["validation"],
    data_collator=instruction_data_collator,
)

print("Instruction Trainer is ready.")

In [ ]:
# ============================================================
# Start instruction fine-tuning
# ============================================================

instruction_train_result = instruction_trainer.train()

In [ ]:
print("Instruction fine-tuning completed.")
print(instruction_train_result)

In [ ]:
# ============================================================
# Save final instruction-tuned LoRA adapter
# ============================================================
# This adapter now contains Stage 1 domain adaptation + Stage 2 instruction tuning.

import os

instruction_adapter_dir = "/content/pharma_tinyllama_instruction_lora_adapter"
os.makedirs(instruction_adapter_dir, exist_ok=True)

instruction_trainer.model.save_pretrained(instruction_adapter_dir)
tokenizer.save_pretrained(instruction_adapter_dir)

print(f"Final instruction-tuned LoRA adapter saved to: {instruction_adapter_dir}")
print(os.listdir(instruction_adapter_dir))

In [ ]:
# ============================================================
# Reload final instruction-tuned adapter for inference
# ============================================================

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

if use_cuda:
    base_model = AutoModelForCausalLM.from_pretrained(
        config.model_name,
        quantization_config=BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        ),
        device_map="auto",
        trust_remote_code=True,
    )
else:
    base_model = AutoModelForCausalLM.from_pretrained(
        config.model_name,
        torch_dtype=torch.float32,
        trust_remote_code=True,
    )

final_instruction_model = PeftModel.from_pretrained(
    base_model,
    instruction_adapter_dir,
)

final_instruction_model.eval()

print("Final instruction-tuned model loaded successfully.")

In [ ]:
# ============================================================
# Instruction-style inference helper
# ============================================================

def build_instruction_prompt(instruction, input_text=""):
    instruction = instruction.strip()
    input_text = input_text.strip()

    if input_text:
        return (
            f"### Instruction:\n{instruction}\n\n"
            f"### Input:\n{input_text}\n\n"
            f"### Response:\n"
        )

    return (
        f"### Instruction:\n{instruction}\n\n"
        f"### Response:\n"
    )


def generate_instruction_response(instruction, input_text="", max_new_tokens=150):
    prompt = build_instruction_prompt(instruction, input_text)

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(final_instruction_model.device)

    with torch.no_grad():
        outputs = final_instruction_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
# ============================================================
# Test instruction-tuned pharma model
# ============================================================

test_questions = [
    "Explain the primary mechanism of action of metformin.",
    "Why can atorvastatin and ezetimibe reduce LDL-C more effectively together?",
    "Summarize the role of lipid nanoparticles in mRNA vaccines.",
    "Why should AI predictions in drug discovery be experimentally validated?",
]

for question in test_questions:
    print("=" * 100)
    print("QUESTION:")
    print(question)

    print("\nMODEL RESPONSE:")
    print(generate_instruction_response(question, max_new_tokens=150))

In [ ]:
# ============================================================
# Merge instruction-tuned LoRA adapter into base model
# ============================================================
# This creates a standalone instruction-tuned model.
# Later, we can use this merged model as the base model for preference tuning.

import os
import gc
import torch

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Path where the final merged instruction-tuned model will be saved.
merged_instruction_model_dir = "/content/pharma_tinyllama_instruction_merged_model"

os.makedirs(merged_instruction_model_dir, exist_ok=True)

# Load the original base model in normal precision for safe merging.
base_model_for_merge = AutoModelForCausalLM.from_pretrained(
    config.model_name,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True,
)

# Load the tokenizer.
tokenizer_for_merge = AutoTokenizer.from_pretrained(
    config.model_name,
    trust_remote_code=True,
)

if tokenizer_for_merge.pad_token is None:
    tokenizer_for_merge.pad_token = tokenizer_for_merge.eos_token

# Attach the final instruction-tuned LoRA adapter.
model_with_instruction_adapter = PeftModel.from_pretrained(
    base_model_for_merge,
    instruction_adapter_dir,
)

# Merge LoRA adapter weights into the base model weights.
merged_instruction_model = model_with_instruction_adapter.merge_and_unload()

# Save the standalone merged model and tokenizer.
merged_instruction_model.save_pretrained(merged_instruction_model_dir)
tokenizer_for_merge.save_pretrained(merged_instruction_model_dir)

print(f"Merged instruction-tuned model saved to: {merged_instruction_model_dir}")

Then preference tuning me merged model as base load karege:


In [ ]:
# ============================================================
# Load merged instruction model as base for preference tuning
# ============================================================

from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

use_cuda = torch.cuda.is_available()

if use_cuda:
    preference_base_model = AutoModelForCausalLM.from_pretrained(
        merged_instruction_model_dir,
        quantization_config=BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        ),
        device_map="auto",
        trust_remote_code=True,
    )

    preference_base_model = prepare_model_for_kbit_training(preference_base_model)

else:
    preference_base_model = AutoModelForCausalLM.from_pretrained(
        merged_instruction_model_dir,
        torch_dtype=torch.float32,
        trust_remote_code=True,
    )

preference_base_model.config.use_cache = False

# Create a new LoRA adapter for preference tuning.
preference_lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

preference_model = get_peft_model(
    preference_base_model,
    preference_lora_config,
)

preference_model.print_trainable_parameters()